In [1]:
from dotenv import load_dotenv
import os
import ollama
import json

## Load environment variables from .env

In [2]:
load_dotenv("../.env")

OLLAMA_CF_CLIENT_ID = os.getenv("OLLAMA_CF_CLIENT_ID")
OLLAMA_CF_CLIENT_SECRET = os.getenv("OLLAMA_CF_CLIENT_SECRET")
OLLAMA_HOST = os.getenv("OLLAMA_HOST")

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

# Model used
MODEL = "qwen3.5:9b"

##  Configuration of Ollama client through cloudflare 

In [3]:
cf_headers = {
    "CF-Access-Client-Id": os.environ.get("OLLAMA_CF_CLIENT_ID"),
    "CF-Access-Client-Secret": os.environ.get("OLLAMA_CF_CLIENT_SECRET"),
}

host = os.environ.get("OLLAMA_HOST")

client = ollama.Client(host=host, headers=cf_headers)

In [4]:
# Test prompt to verify
response = client.chat(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say Hi to me and ask how my day was"}
    ]
)

print("Ollama connection test\n")
print(response.message.content)

Ollama connection test

Hi there! How was your day? 😊


## Mock Data for testing

In [5]:
mock_input = {
    "user_id": "123",
    "sentiment_label": "Anxiety",
    "mood_trend": "improving",
    "selected_tags": ["Calm & Peace", "Music"] 
}

## Prompt Engineering and query generation

In [12]:
def build_search_query(mock_input):
    sentiment = mock_input["sentiment_label"]
    mood_trend = mock_input["mood_trend"]
    tags = ", ".join(mock_input["selected_tags"])  # ["Calm & Peace", "Music"] to "Calm & Peace, Music"

    prompt = f"""

    You are a mental wellness YouTube video recommendation assistant.
Your sole purpose is to find videos that are safe, appropriate, and genuinely 
helpful for someone based on their current emotional state and preferences.

Current mood: {sentiment}
Mood trend: {mood_trend}
User preferred video types: {tags}

Mood Rules

Suicidal:
    Only recommend extremely gentle and grounding content.
    Every query must feel safe, warm, and non-triggering.
    Never suggest anything emotionally intense, heavy, or overwhelming.
    Themes: "you are not alone", calm nature visuals, soft reassuring voices.

Depression:
    Recommend soft, comforting, and quietly hopeful content.
    Avoid high energy content and toxic positivity at all costs.
    Themes: cozy vlogs, soft spoken advice, personal healing journeys.

Anxiety:
    Recommend calming content that slows the mind down.
    Target nervous system regulation, breathing, and present moment awareness.
    Avoid anything fast paced, loud, or overstimulating.
    Themes: breathwork, guided meditation, anxiety education, calm music.

Stress:
    Recommend light and gently relieving content.
    Avoid complex, heavy, or problem solving content.
    Themes: satisfying videos, light documentaries, stress relief techniques.

Normal:
    Recommend engaging, motivational, and growth oriented content.
    Avoid emotionally heavy or overly therapeutic content.
    Themes: productivity, interesting science, motivational talks.

Trend Rules

Improving:
    Lean slightly more positive and uplifting across all queries.
    Acceptable to include mildly energizing or inspiring content.

Declining:
    Apply extra gentleness and emotional care to every query.
    Prioritize emotional safety above everything else including engagement.

Tag Rules

The user has selected their preferred video types: {tags}
These preferences must be reflected across the generated queries.

Calm and Peace: meditation, nature sounds, ambient music, sleep content.
Educational: psychology explainers, TED style talks, mind related science.
Documentary: short human interest documentaries, real life personal journeys.
Science: neuroscience, mind body connection, biology of emotions.
Music: lofi playlists, instrumental music, mood based music.

Always blend the mood rules and tag preferences naturally into each query.
Never generate content that could lead to harmful or triggering results
for someone in a vulnerable emotional state. When in doubt, choose the gentler option.

Output

Generate exactly 5 YouTube search queries based on everything above.
Each query must be a maximum of 8 words and sound like something a real person would search.
Return only a JSON array of 5 strings.
Example: ["query one", "query two", "query three", "query four", "query five"]
No explanation. No extra text. No markdown. Just the JSON array.
"""

    response = client.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.message.content.strip()

    queries = json.loads(raw)
    return queries

queries = build_search_query(mock_input)
print("Generated search queries \n")
for i, q in enumerate(queries, 1):
    print(f"{i}. {q}")

Generated search queries 

1. calm music for anxiety relief breathing exercise
2. gentle nature sounds for calming anxious thoughts
3. slow lofi beats to relax racing mind
4. soft guided meditation for deep anxiety relief
5. gentle hopeful music to ease anxious mind
